In [ ]:

import numpy as np
from qiskit import QuantumCircuit
import matplotlib.pyplot as plt

import numpy as np


def binary_to_integer(vOutput): # binary to integer
    vPows = 2**np.array(range(len(vOutput)))  
    vPows = vPows[::-1]                       #MSB...LSB
    return np.dot(vOutput, vPows)             #Integer, Dot product

vOutput = np.array((1, 0, 1, 0))
print(binary_to_integer(vOutput)) 

A. Auxiliary Block

In [92]:
from qiskit import QuantumCircuit

def set_classical_value_to_register(qc: QuantumCircuit, x: int, wires):
    n = len(wires)
    bits = [(x >> i) & 1 for i in range(n)][::-1]
    for b, w in zip(bits, wires):
        if b == 1:
            qc.x(w)

def copy_register(qc: QuantumCircuit, src_wires, dst_wires):
    for s, d in zip(src_wires, dst_wires):
        qc.cx(s, d)

def clear_register_with_copy(qc: QuantumCircuit, src_wires, dst_wires):
    for s, d in zip(src_wires, dst_wires):
        qc.cx(s, d)

                                 
 q_0: ───────────■───────────────
      ┌───┐      │               
 q_1: ┤ X ├──────┼───────────────
      └───┘      │               
 q_2: ───────────┼───────────────
      ┌───┐┌─────┴─────┐┌─┐      
 q_3: ┤ X ├┤0          ├┤M├──────
      └───┘│           │└╥┘┌─┐   
 q_4: ─────┤1          ├─╫─┤M├───
           │           │ ║ └╥┘┌─┐
 q_5: ─────┤2          ├─╫──╫─┤M├
           │           │ ║  ║ └╥┘
 q_6: ─────┤3          ├─╫──╫──╫─
           │           │ ║  ║  ║ 
 q_7: ─────┤4          ├─╫──╫──╫─
           │           │ ║  ║  ║ 
 q_8: ─────┤5          ├─╫──╫──╫─
           │           │ ║  ║  ║ 
 q_9: ─────┤6          ├─╫──╫──╫─
           │           │ ║  ║  ║ 
q_10: ─────┤7  mult(4) ├─╫──╫──╫─
           │           │ ║  ║  ║ 
q_11: ─────┤8          ├─╫──╫──╫─
           │           │ ║  ║  ║ 
q_12: ─────┤9          ├─╫──╫──╫─
           │           │ ║  ║  ║ 
q_13: ─────┤10         ├─╫──╫──╫─
           │           │ ║  ║  ║ 
q_14: ─────┤11

B. In-place Quantum-Classical Adder

In [93]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator
from qiskit import transpile

def _phase_add(qc: QuantumCircuit, regs, k: int):  # Fourier basis: a <- a + k
    for j, q in enumerate(regs):
        qc.rz(k * np.pi / (2**j), q)

def add_const(qc: QuantumCircuit, a, k: int, in_std_basis: bool = True):
    n = len(a)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    _phase_add(qc, a, k)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

def add_const_mod_N(qc: QuantumCircuit, a, aux, k: int, N: int): #_phase_add mod N
    n = len(a)

    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True) #In Fourier basis

    # Step 1: add k
    _phase_add(qc, a, k)

    # Step 2: subtract N
    _phase_add(qc, a, -N)

    # Step 3: MSB-computational basis
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)
    qc.cx(a[-1], aux[0])      # MSB index
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)

    # Step 4: Add N, conditionally
    for j, q in enumerate(a):
        qc.crz(N * np.pi / (2**j), aux[0], q)

    # Step 5: Inverse of Step 3
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)
    qc.cx(a[-1], aux[0])
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)

    # Computational basis
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

# Circuit
def circuit_add_in_k_N(a_val, k, N, wires_a, aux_idx, n_qubits=10):
    qc = QuantumCircuit(n_qubits, len(wires_a))
    q = qc.qubits
    a = [q[i] for i in wires_a]
    aux = [q[aux_idx]]

    # for a[0]-LSB
    for bit_pos, qb in enumerate(a):
        if (a_val >> bit_pos) & 1:
            qc.x(qb)

    add_const_mod_N(qc, a, aux, k, N)

    # Measure 'a' classical
    for i, qb in enumerate(a):
        qc.measure(qb, i)

    return qc

#Example: value, circuit
N = 7
a_val = 4
k = 5
qc = circuit_add_in_k_N(a_val, k, N, wires_a=[1,2,3], aux_idx=4, n_qubits=10)


backend = AerSimulator()
res = backend.run(transpile(qc, backend), shots=1, memory=True).result()
print(f"{a_val} + {k} mod {N} = {int(res.get_memory()[0], 2)}")

print(qc.draw('text'))




4 + 5 mod 7 = 2
                                                                           »
q_0: ──────────────────────────────────────────────────────────────────────»
          ┌──────┐ ┌────────┐  ┌─────────┐ ┌───────┐     ┌──────┐┌────────┐»
q_1: ─────┤0     ├─┤ Rz(5π) ├──┤ Rz(-7π) ├─┤0      ├─────┤0     ├┤ Rz(7π) ├»
          │      │┌┴────────┴┐┌┴─────────┴┐│       │     │      │└───┬────┘»
q_2: ─────┤1 QFT ├┤ Rz(5π/2) ├┤ Rz(-7π/2) ├┤1 IQFT ├─────┤1 QFT ├────┼─────»
     ┌───┐│      │├──────────┤├───────────┤│       │     │      │    │     »
q_3: ┤ X ├┤2     ├┤ Rz(5π/4) ├┤ Rz(-7π/4) ├┤2      ├──■──┤2     ├────┼─────»
     └───┘└──────┘└──────────┘└───────────┘└───────┘┌─┴─┐└──────┘    │     »
q_4: ───────────────────────────────────────────────┤ X ├────────────■─────»
                                                    └───┘                  »
q_5: ──────────────────────────────────────────────────────────────────────»
                                                            

C. Out-place Quantum-Classic Adder 

In [94]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator
from qiskit import transpile


def _phase_add(qc: QuantumCircuit, regs, k: int):  #Fourier-basis: a <- a + k
    for j, q in enumerate(regs):
        qc.rz(k * np.pi / (2**j), q)

def add_const(qc: QuantumCircuit, a, k: int, in_std_basis: bool = True):
    n = len(a)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    _phase_add(qc, a, k)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

def add_const_mod_N(qc: QuantumCircuit, a, aux, k: int, N: int):
    n = len(a)
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)              # QFT
    _phase_add(qc, a, k)                                                    # +k
    _phase_add(qc, a, -N)                                                   # -N
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)    # !
    qc.cx(a[-1], aux[0])
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    for j, q in enumerate(a):                                               # +N if!
        qc.crz(N * np.pi / (2**j), aux[0], q)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)    # !!
    qc.cx(a[-1], aux[0])
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)    # IQFT

# Adder
def add_out_k_N(qc: QuantumCircuit, a, res, aux, k: int, N: int):
    if len(aux) < 2:
        raise ValueError("2 ancillas needed, 'aux'")

    # 1) CNOT:
    for i in range(len(res)):
        qc.cx(a[i], res[i])

    # 2) MSB - aux[0]
    r = res + [aux[0]]

    # 3) Modular add 'r' ---> aux[1]
    add_const_mod_N(qc, r, [aux[1]], k, N)

# Circuit
def circuit_add_out_k_N(a_val, k, N, wires_a, wires_res, wires_aux, n_qubits=8):
    qc = QuantumCircuit(n_qubits, len(wires_res))
    q = qc.qubits
    a   = [q[i] for i in wires_a]
    res = [q[i] for i in wires_res]
    aux = [q[i] for i in wires_aux]  # len>=2

    # |a> = a_val  (a[0]=LSB)
    for bit_pos, qb in enumerate(a):
        if (a_val >> bit_pos) & 1:
            qc.x(qb)

    add_out_k_N(qc, a, res, aux, k, N)

    # Measure 'res'
    for i, qb in enumerate(res):
        qc.measure(qb, i)

    return qc

#Example
N = 7
a_val = 2
k = 5
wires_a   = [0, 1, 2]
wires_res = [3, 4, 5]
wires_aux = [6, 7]
qc = circuit_add_out_k_N(a_val, k, N, wires_a, wires_res, wires_aux, n_qubits=8)

backend = AerSimulator()
res = backend.run(transpile(qc, backend), shots=1, memory=True).result()

print(f"{a_val} + {k} mod {N} = {int(res.get_memory()[0], 2)}")

print(qc.draw('text'))

2 + 5 mod 7 = 0
                                                                           »
q_0: ───────■──────────────────────────────────────────────────────────────»
     ┌───┐  │                                                              »
q_1: ┤ X ├──┼────■─────────────────────────────────────────────────────────»
     └───┘  │    │                                                         »
q_2: ──■────┼────┼─────────────────────────────────────────────────────────»
       │  ┌─┴─┐  │  ┌──────┐ ┌────────┐  ┌─────────┐ ┌───────┐     ┌──────┐»
q_3: ──┼──┤ X ├──┼──┤0     ├─┤ Rz(5π) ├──┤ Rz(-7π) ├─┤0      ├─────┤0     ├»
       │  └───┘┌─┴─┐│      │┌┴────────┴┐┌┴─────────┴┐│       │     │      │»
q_4: ──┼───────┤ X ├┤1     ├┤ Rz(5π/2) ├┤ Rz(-7π/2) ├┤1      ├─────┤1     ├»
     ┌─┴─┐     └───┘│  QFT │├──────────┤├───────────┤│  IQFT │     │  QFT │»
q_5: ┤ X ├──────────┤2     ├┤ Rz(5π/4) ├┤ Rz(-7π/4) ├┤2      ├─────┤2     ├»
     └───┘          │      │├──────────┤├───────────┤│      

D.  In-place Quantum-Quantum Adder

In [95]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator
from qiskit import transpile

def _phase_add(qc: QuantumCircuit, regs, k: int):  #Fourier basis: a <- a + k
    for j, q in enumerate(regs):
        qc.rz(k * np.pi / (2**j), q)

def add_const(qc: QuantumCircuit, a, k: int, in_std_basis: bool = True):
    n = len(a)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    _phase_add(qc, a, k)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

def add_const_mod_N(qc: QuantumCircuit, a, aux, k: int, N: int):
    n = len(a)
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    _phase_add(qc, a, k)
    _phase_add(qc, a, -N)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)
    qc.cx(a[-1], aux[0])
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    for j, q in enumerate(a):
        qc.crz(N * np.pi / (2**j), aux[0], q)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)
    qc.cx(a[-1], aux[0])
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

## +k (controlled modular) in Fourier basis
def _mod_add_N_fourier_gate(r_len: int, k: int, N: int):
    sub = QuantumCircuit(r_len + 1, name=f"modAddN({k})")
    r = list(range(r_len))
    flag = r_len 
# LSB r[0], MSB r[-1]

## -N in Fourier basis
    for j in range(r_len):
        sub.rz(k * np.pi / (2**j), r[j])
    for j in range(r_len):
        sub.rz(-N * np.pi / (2**j), r[j])

# !
    sub.compose(QFT(r_len, do_swaps=False).inverse(), qubits=r, inplace=True)
    sub.cx(r[-1], flag)
    sub.compose(QFT(r_len, do_swaps=False), qubits=r, inplace=True)

    # +N iff !
    for j in range(r_len):
        sub.crz(N * np.pi / (2**j), flag, r[j])

    # !!
    sub.compose(QFT(r_len, do_swaps=False).inverse(), qubits=r, inplace=True)
    sub.cx(r[-1], flag)
    sub.compose(QFT(r_len, do_swaps=False), qubits=r, inplace=True)
    return sub.to_gate()

# Adder
def add_in_N(qc: QuantumCircuit, a, b, aux1, aux2, N: int):
    r = b + [aux1[0]] # MSB +
    n_r = len(r)

    # 1) Fourier base (r)
    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)
    
    # 2) Controlled phase shifts
    for i, ctrl in enumerate(a):
        value = (1 << i) % N
        for j, q in enumerate(r):
            qc.crz(value * np.pi / (2**j), ctrl, q)

    # 3) -N
    _phase_add(qc, r, -N)  # r-N (in Fourier)
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)
    qc.cx(r[-1], aux2[0])  # ! MSB of r
    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)

    # +N, conditional
    for j, q in enumerate(r):
        qc.crz(N * np.pi / (2**j), aux2[0], q)

    # !!
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)
    qc.cx(r[-1], aux2[0])
    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)

    # 4) r computational basis
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)


# Circuit
def circuit_add_in_N(a_val, b_val, N, wires_a, wires_b, aux1, aux2, n_qubits=10):
    qc = QuantumCircuit(n_qubits, len(wires_b))
    q = qc.qubits
    a   = [q[i] for i in wires_a]
    b   = [q[i] for i in wires_b]
    aux1q = [q[i] for i in aux1]   # qubit: MSB+
    aux2q = [q[i] for i in aux2]   # qubit: !

    # |a> and |b> (LSB=[0])
    for bit_pos, qb in enumerate(a):
        if (a_val >> bit_pos) & 1: qc.x(qb)
    for bit_pos, qb in enumerate(b):
        if (b_val >> bit_pos) & 1: qc.x(qb)

    add_in_N(qc, a, b, aux1q, aux2q, N)

    # Measure b
    for i, qb in enumerate(b):
        qc.measure(qb, i)

    return qc

#Example
N = 7
a_val = 3
b_val = 5
wires_a = [0,1,2]
wires_b = [3,4,5]
aux1    = [6]    # b, MSB+
aux2    = [7]    # ancilla !
qc = circuit_add_in_N(a_val, b_val, N, wires_a, wires_b, aux1, aux2, n_qubits=10)

backend = AerSimulator()
res = backend.run(transpile(qc, backend), shots=1, memory=True).result()
print(f"{a_val} + {b_val} mod {N} = {int(res.get_memory()[0], 2)}")

print(qc.draw('text'))

3 + 5 mod 7 = 1
     ┌───┐                                                                    »
q_0: ┤ X ├────────────■─────────■────────────────────■────────────────────────»
     ├───┤            │         │                    │                        »
q_1: ┤ X ├────────────┼─────────┼─────────■──────────┼─────────■──────────────»
     └───┘            │         │         │          │         │              »
q_2: ─────────────────┼─────────┼─────────┼──────────┼─────────┼────────■─────»
     ┌───┐┌──────┐┌───┴───┐     │     ┌───┴────┐     │         │    ┌───┴────┐»
q_3: ┤ X ├┤0     ├┤ Rz(π) ├─────┼─────┤ Rz(2π) ├─────┼─────────┼────┤ Rz(4π) ├»
     └───┘│      │└───────┘┌────┴────┐└────────┘     │     ┌───┴───┐└────────┘»
q_4: ─────┤1     ├─────────┤ Rz(π/2) ├───────────────┼─────┤ Rz(π) ├──────────»
     ┌───┐│  QFT │         └─────────┘          ┌────┴────┐└───────┘          »
q_5: ┤ X ├┤2     ├──────────────────────────────┤ Rz(π/4) ├───────────────────»
     └───┘│      │      

E. Outplace Quantum-Quantum Adder

In [96]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator
from qiskit import transpile

def _phase_add(qc: QuantumCircuit, regs, k: int):           # Fourier basis: regs <- regs + k
    for j, q in enumerate(regs):                            # LSB@regs[0]
        qc.rz(k * np.pi / (2**j), q)

def add_const(qc: QuantumCircuit, a, k: int, in_std_basis: bool = True):
    n = len(a)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    _phase_add(qc, a, k)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

def add_const_mod_N(qc: QuantumCircuit, a, aux, k: int, N: int):
    n = len(a)
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    _phase_add(qc, a, k)
    _phase_add(qc, a, -N)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)
    qc.cx(a[-1], aux[0])                                    
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    for j, q in enumerate(a):
        qc.crz(N * np.pi / (2**j), aux[0], q)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)
    qc.cx(a[-1], aux[0])
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

#b <- (a + b) mod N 
def add_in_N(qc: QuantumCircuit, a, b, aux1, aux2, N: int):
    r   = b + [aux1[0]]                                 
    n_r = len(r)

    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)
    for i, ctrl in enumerate(a):                          #r, Fourier basis
        value = (1 << i) % N
        for j, q in enumerate(r):
            qc.crz(value * np.pi / (2**j), ctrl, q)

    #-N
    _phase_add(qc, r, -N)                                  
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)
    ## ! (N)
    qc.cx(r[-1], aux2[0])                                  
    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)
    
    # !! (N)
    for j, q in enumerate(r):                              
        qc.crz(N * np.pi / (2**j), aux2[0], q)
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)
    qc.cx(r[-1], aux2[0])
    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)

#res <- (a + b) mod N -out
def add_out_N(qc: QuantumCircuit, a, b, res, aux1, aux2, N: int):
    for i in range(len(b)):                                 
        qc.cx(b[i], res[i])
    add_in_N(qc, a, res, aux1, aux2, N)                     

#Circuit
def circuit_add_out_N(a_val, b_val, N, wires_a, wires_b, wires_res, aux1, aux2, n_qubits=11):
    qc = QuantumCircuit(n_qubits, len(wires_res))
    q  = qc.qubits
    a   = [q[i] for i in wires_a]
    b   = [q[i] for i in wires_b]
    res = [q[i] for i in wires_res]
    aux1q = [q[i] for i in aux1]    
    aux2q = [q[i] for i in aux2]   

    # |a>, |b> (LSB [0])
    a_val %= N; b_val %= N
    for bit_pos, qb in enumerate(a):
        if (a_val >> bit_pos) & 1: qc.x(qb)
    for bit_pos, qb in enumerate(b):
        if (b_val >> bit_pos) & 1: qc.x(qb)

    add_out_N(qc, a, b, res, aux1q, aux2q, N)

#MEasure
    for i, qb in enumerate(res):            
        qc.measure(qb, i)
    return qc

#Example 
N = 7
a_val = 3
b_val = 5
wires_a   = [0,1,2]
wires_b   = [3,4,5]
wires_res = [6,7,8]
aux1      = [9]         
aux2      = [10]        
qc = circuit_add_out_N(a_val, b_val, N, wires_a, wires_b, wires_res, aux1, aux2, n_qubits=11)

backend = AerSimulator()
res = backend.run(transpile(qc, backend), shots=1, memory=True).result()
print(f"{a_val} + {b_val} mod {N} = {int(res.get_memory()[0], 2)}")
print(qc.draw('text'))


3 + 5 mod 7 = 1
      ┌───┐                                                                »
 q_0: ┤ X ├───────────────────────────■─────────■────────────────────■─────»
      ├───┤                           │         │                    │     »
 q_1: ┤ X ├───────────────────────────┼─────────┼─────────■──────────┼─────»
      └───┘                           │         │         │          │     »
 q_2: ────────────────────────────────┼─────────┼─────────┼──────────┼─────»
      ┌───┐                           │         │         │          │     »
 q_3: ┤ X ├───────■───────────────────┼─────────┼─────────┼──────────┼─────»
      └───┘       │                   │         │         │          │     »
 q_4: ───────■────┼───────────────────┼─────────┼─────────┼──────────┼─────»
      ┌───┐  │    │                   │         │         │          │     »
 q_5: ┤ X ├──┼────┼────■──────────────┼─────────┼─────────┼──────────┼─────»
      └───┘  │  ┌─┴─┐  │  ┌──────┐┌───┴───┐     │     ┌───┴─

F. Out-place Quantum-Classic Multiplier

In [97]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator
from qiskit import transpile


def _phase_add(qc: QuantumCircuit, regs, k: int):  #Fourier basis: regs <- regs + k
    for j, q in enumerate(regs):
        qc.rz(k * np.pi / (2**j), q)

def add_const(qc: QuantumCircuit, a, k: int, in_std_basis: bool = True):
    n = len(a)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    _phase_add(qc, a, k)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

def add_const_mod_N(qc: QuantumCircuit, a, aux, k: int, N: int):
    n = len(a)
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    _phase_add(qc, a, k)
    _phase_add(qc, a, -N)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)
    qc.cx(a[-1], aux[0])
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    for j, q in enumerate(a):
        qc.crz(N * np.pi / (2**j), aux[0], q)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)
    qc.cx(a[-1], aux[0])
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

# b <- (k*a + b) mod N
def mult_out_k_N(qc: QuantumCircuit, a, b, aux, k: int, N: int):
    if len(aux) < 2:
        raise ValueError("2 ancillas needed 'aux'")

    # register, r
    r = b + [aux[0]]
    n_r = len(r)

    # 1) r in Fourier basis
    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)

    # 2) k*(2^i)
    for i, ctrl in enumerate(a):                # a[0]=LSB
        value = (k * (1 << i)) % N
        for j, q in enumerate(r):
            qc.crz(value * np.pi / (2**j), ctrl, q)

    # 3) -N in Fourirer 
    _phase_add(qc, r, -N)                           
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)
    qc.cx(r[-1], aux[1])                            # !
    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)

    # +N, conditional
    for j, q in enumerate(r):
        qc.crz(N * np.pi / (2**j), aux[1], q)

    # !!
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)
    qc.cx(r[-1], aux[1])
    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)

    # 4) Computational basis
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)

# Circuit
def circuit_mult_out_k_N(a_val, b_val, k, N, wires_a, wires_b, wires_aux, n_qubits=8):
    qc = QuantumCircuit(n_qubits, len(wires_b))
    q = qc.qubits
    a   = [q[i] for i in wires_a]
    b   = [q[i] for i in wires_b]
    aux = [q[i] for i in wires_aux]  # 2 qubits needed

    #|a>, |b| (LSB@[0])
    for bit_pos, qb in enumerate(a):
        if (a_val >> bit_pos) & 1: qc.x(qb)
    for bit_pos, qb in enumerate(b):
        if (b_val >> bit_pos) & 1: qc.x(qb)

    mult_out_k_N(qc, a, b, aux, k, N)

    # Measure b
    for i, qb in enumerate(b):
        qc.measure(qb, i)

    return qc

#Example
N = 7
a_val = 4
k = 2
b_val = 3
wires_a = [0,1,2]
wires_b = [3,4,5]
wires_aux = [6,7]
qc = circuit_mult_out_k_N(a_val, b_val, k, N, wires_a, wires_b, wires_aux, n_qubits=8)

backend = AerSimulator()
res = backend.run(transpile(qc, backend), shots=1, memory=True).result()
print(f"{a_val} * {k} + {b_val} mod {N} = {int(res.get_memory()[0], 2)}")

print(qc.draw('text'))



4 * 2 + 3 mod 7 = 4
                                                                             »
q_0: ─────────────────■─────────■───────────────────■────────────────────────»
                      │         │                   │                        »
q_1: ─────────────────┼─────────┼────────■──────────┼─────────■──────────────»
     ┌───┐            │         │        │          │         │              »
q_2: ┤ X ├────────────┼─────────┼────────┼──────────┼─────────┼─────────■────»
     ├───┤┌──────┐┌───┴────┐    │    ┌───┴────┐     │         │     ┌───┴───┐»
q_3: ┤ X ├┤0     ├┤ Rz(2π) ├────┼────┤ Rz(4π) ├─────┼─────────┼─────┤ Rz(π) ├»
     ├───┤│      │└────────┘┌───┴───┐└────────┘     │     ┌───┴────┐└───────┘»
q_4: ┤ X ├┤1     ├──────────┤ Rz(π) ├───────────────┼─────┤ Rz(2π) ├─────────»
     └───┘│  QFT │          └───────┘          ┌────┴────┐└────────┘         »
q_5: ─────┤2     ├─────────────────────────────┤ Rz(π/2) ├───────────────────»
          │      │              

G. In-place Quantum-classical multiplier

In [98]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator
from qiskit import transpile

def _phase_add(qc: QuantumCircuit, regs, k: int):  #Fourier basis: regs <- regs + k
    for j, q in enumerate(regs):
        qc.rz(k * np.pi / (2**j), q)

def add_const(qc: QuantumCircuit, a, k: int, in_std_basis: bool = True):
    n = len(a)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    _phase_add(qc, a, k)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

def add_const_mod_N(qc: QuantumCircuit, a, aux, k: int, N: int):
    n = len(a)
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    _phase_add(qc, a, k)
    _phase_add(qc, a, -N)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)
    qc.cx(a[-1], aux[0])
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    for j, q in enumerate(a):
        qc.crz(N * np.pi / (2**j), aux[0], q)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)
    qc.cx(a[-1], aux[0])
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

# b <- (k*a + b) mod N
def mult_out_k_N(qc: QuantumCircuit, a, b, aux, k: int, N: int):
    if len(aux) < 2:
        raise ValueError("2 ancillas needed 'aux'")

    #Register, r
    r = b + [aux[0]]
    n_r = len(r)

    # 1) r in Fourier basis
    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)

    # 2) k*(2^i)
    for i, ctrl in enumerate(a):               # a[0]=LSB
        value = (k * (1 << i)) % N
        for j, q in enumerate(r):
            qc.crz(value * np.pi / (2**j), ctrl, q)

    # 3) -N
    _phase_add(qc, r, -N)
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)
    qc.cx(r[-1], aux[1])
    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)
    # +N, conditional
    for j, q in enumerate(r):
        qc.crz(N * np.pi / (2**j), aux[1], q)
    # !!
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)
    qc.cx(r[-1], aux[1])
    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)

    # 4) Computational basis
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)

# a <- (k * a) mod N 
def mult_in_k_N(qc: QuantumCircuit, a, support, aux, k: int, N: int):
    if len(aux) < 2 or len(a) != len(support):
        raise ValueError(" Error; ancillas or len(a), len(support)'")

    inv_k = pow(k, N-2, N)   # (For N=prime)
    if (inv_k * k) % N != 1:
        raise ValueError(f"No inverse for k={k} modulo N={N}.")

    # Step 1: Support <- (k*a + support) mod N
    mult_out_k_N(qc, a, support, aux, k, N)

    # Step 2: swap registers, for 'a' == k*a mod N
    for i in range(len(a)):
        qc.swap(a[i], support[i])

    # Step 3: Clear support
    mult_out_k_N(qc, a, support, aux, (N - inv_k) % N, N)

# Circuit
def circuit_mult_in_k_N(a_val, k, N, wires_a, wires_support, wires_aux, n_qubits=8):
    qc = QuantumCircuit(n_qubits, len(wires_a))
    q = qc.qubits
    a        = [q[i] for i in wires_a]
    support  = [q[i] for i in wires_support]
    aux      = [q[i] for i in wires_aux]   # 2 qubits needed

    # |a> = a_val (LSB@[0
    a_val %= N
    for bit_pos, qb in enumerate(a):
        if (a_val >> bit_pos) & 1:
            qc.x(qb)

    mult_in_k_N(qc, a, support, aux, k % N, N)

    # measure a
    for i, qb in enumerate(a):
        qc.measure(qb, i)

    return qc

# Example
N = 5
a_val = 3
k = 3
wires_a       = [0,1,2]
wires_support = [3,4,5]
wires_aux     = [6,7]
qc = circuit_mult_in_k_N(a_val, k, N, wires_a, wires_support, wires_aux, n_qubits=8)

backend = AerSimulator()
res = backend.run(transpile(qc, backend), shots=1, memory=True).result()
print(f"{a_val} * {k} mod {N} = {int(res.get_memory()[0], 2)}")


print(qc.draw('text'))

3 * 3 mod 5 = 4
      ┌───┐                                                                  »
q_0: ─┤ X ├──────■──────────■────────────────────■───────────────────────────»
      ├───┤      │          │                    │                           »
q_1: ─┤ X ├──────┼──────────┼──────────■─────────┼───────────■───────────────»
      └───┘      │          │          │         │           │               »
q_2: ────────────┼──────────┼──────────┼─────────┼───────────┼─────────■─────»
     ┌──────┐┌───┴────┐     │      ┌───┴───┐     │           │     ┌───┴────┐»
q_3: ┤0     ├┤ Rz(3π) ├─────┼──────┤ Rz(π) ├─────┼───────────┼─────┤ Rz(2π) ├»
     │      │└────────┘┌────┴─────┐└───────┘     │      ┌────┴────┐└────────┘»
q_4: ┤1     ├──────────┤ Rz(3π/2) ├──────────────┼──────┤ Rz(π/2) ├──────────»
     │  QFT │          └──────────┘         ┌────┴─────┐└─────────┘          »
q_5: ┤2     ├───────────────────────────────┤ Rz(3π/4) ├─────────────────────»
     │      │                       

H. Outplace modular quantum–quantum multiplier

In [99]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator
from qiskit import transpile

def _phase_add(qc: QuantumCircuit, regs, k: int):  #Fourier basis: regs <- regs + k
    for j, q in enumerate(regs):
        qc.rz(k * np.pi / (2**j), q)

def add_const(qc: QuantumCircuit, a, k: int, in_std_basis: bool = True):
    n = len(a)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    _phase_add(qc, a, k)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

def add_const_mod_N(qc: QuantumCircuit, a, aux, k: int, N: int):
    n = len(a)
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    _phase_add(qc, a, k)
    _phase_add(qc, a, -N)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)
    qc.cx(a[-1], aux[0])
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    for j, q in enumerate(a):
        qc.crz(N * np.pi / (2**j), aux[0], q)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)
    qc.cx(a[-1], aux[0])
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

# res <- (a * b) mod N
def mult_out_N(qc: QuantumCircuit, a, b, res, aux, N: int):
    if len(aux) < 2:
        raise ValueError("2 ancillas needed 'aux'.")

    # Register, r
    r = res + [aux[0]]
    n_r = len(r)

    # 1) r in Fourier basis
    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)

    # 2) (a[i] & b[j]) * 2^(i+j) in Fourier basis
    #    multi-controlled RZ
    for i, ci in enumerate(a):            # a[0]==LSB
        for j, cj in enumerate(b):        # b[0]==LSB
            value = (1 << (i + j)) % N
            if value == 0:
                continue
            for m, q in enumerate(r):
                theta = value * np.pi / (2**m)
                qc.mcrz(theta, [ci, cj], q)

    # 3) -N
    _phase_add(qc, r, -N)
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)
    qc.cx(r[-1], aux[1])                     # flag underflow via MSB of r
    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)
    # !N
    for m, q in enumerate(r):
        qc.crz(N * np.pi / (2**m), aux[1], q)
    # !!(N)
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)
    qc.cx(r[-1], aux[1])
    qc.compose(QFT(n_r, do_swaps=False), qubits=r, inplace=True)

    # 4)Computational basis (r)
    qc.compose(QFT(n_r, do_swaps=False).inverse(), qubits=r, inplace=True)

# Circuit
def circuit_mult_out_N(a_val, b_val, N, wires_a, wires_b, wires_res, wires_aux, n_qubits=11):
    qc = QuantumCircuit(n_qubits, len(wires_res))
    q = qc.qubits
    a   = [q[i] for i in wires_a]
    b   = [q[i] for i in wires_b]
    res = [q[i] for i in wires_res]
    aux = [q[i] for i in wires_aux]  # 2 qubits needed 

    # |a>, |b> (LSB@ [0])
    a_val %= N
    b_val %= N
    for bit_pos, qb in enumerate(a):
        if (a_val >> bit_pos) & 1: qc.x(qb)
    for bit_pos, qb in enumerate(b):
        if (b_val >> bit_pos) & 1: qc.x(qb)

    mult_out_N(qc, a, b, res, aux, N)

    # Measure
    for i, qb in enumerate(res):
        qc.measure(qb, i)

    return qc

#Example
N = 7
a_val = 5
b_val = 5
wires_a   = [0,1,2]
wires_b   = [3,4,5]
wires_res = [6,7,8]
wires_aux = [9,10]
qc = circuit_mult_out_N(a_val, b_val, N, wires_a, wires_b, wires_res, wires_aux, n_qubits=11)

backend = AerSimulator()
res = backend.run(transpile(qc, backend), shots=1, memory=True).result()
print(f"{a_val} * {b_val} mod {N} = {int(res.get_memory()[0], 2)}")

print(qc.draw('text'))



5 * 5 mod 7 = 4
       ┌───┐                                                         »
 q_0: ─┤ X ├────■────────────────────────────────■────────────────■──»
       └───┘    │                                │                │  »
 q_1: ──────────┼────────────────────────────────┼────────────────┼──»
       ┌───┐    │                                │                │  »
 q_2: ─┤ X ├────┼────────────────────────────────┼────────────────┼──»
       ├───┤    │                                │                │  »
 q_3: ─┤ X ├────┼────────────────■───────────────┼────────────────┼──»
       └───┘    │                │               │                │  »
 q_4: ──────────┼────────────────┼───────────────┼────────────────┼──»
       ┌───┐    │                │               │                │  »
 q_5: ─┤ X ├────┼────────────────┼───────────────┼────────────────┼──»
      ┌┴───┴─┐┌─┴─┐┌──────────┐┌─┴─┐┌─────────┐┌─┴─┐┌──────────┐  │  »
 q_6: ┤0     ├┤ X ├┤ Rz(-π/4) ├┤ X ├┤ Rz(π/4) ├┤ X ├┤ Rz(-π/4

J. Out-place modular exponentiation

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT
from qiskit_aer import AerSimulator
from qiskit import transpile


def _phase_add(qc: QuantumCircuit, regs, k: int):           #Fourier basis: regs <- regs + k
    for j, q in enumerate(regs):
        qc.rz(k * np.pi / (2**j), q)

def add_const(qc: QuantumCircuit, a, k: int, in_std_basis: bool = True):
    n = len(a)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    _phase_add(qc, a, k)
    if in_std_basis:
        qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

def add_const_mod_N(qc: QuantumCircuit, a, aux, k: int, N: int):
    n = len(a)
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    _phase_add(qc, a, k)
    _phase_add(qc, a, -N)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)
    qc.cx(a[-1], aux[0])           
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    for j, q in enumerate(a):
        qc.crz(N * np.pi / (2**j), aux[0], q)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)
    qc.cx(a[-1], aux[0])
    qc.compose(QFT(n, do_swaps=False), qubits=a, inplace=True)
    qc.compose(QFT(n, do_swaps=False).inverse(), qubits=a, inplace=True)

# b <- (b + k*a) mod N  (uncontrolled)
def mad_out_k_N(qc: QuantumCircuit, a, b, aux, k: int, N: int):
    if len(aux) < 2:
        raise ValueError("2 ancillas needed")
    extra, flag = aux[0], aux[1]
    r = b + [extra]
    m = len(r)

    k %= N
    if k == 0:
        return

    # k*a in Fourier
    qc.compose(QFT(m, do_swaps=False), qubits=r, inplace=True)
    for i, ai in enumerate(a):
        add_val = (k * (1 << i)) % N
        if add_val == 0: 
            continue
        for j, q in enumerate(r):
            qc.crz(add_val * np.pi / (2**j), ai, q)
    qc.compose(QFT(m, do_swaps=False).inverse(), qubits=r, inplace=True)

    # -N
    qc.compose(QFT(m, do_swaps=False), qubits=r, inplace=True)
    _phase_add(qc, r, -N)
    qc.compose(QFT(m, do_swaps=False).inverse(), qubits=r, inplace=True)  #MSB
    qc.cx(r[-1], flag)
    qc.compose(QFT(m, do_swaps=False), qubits=r, inplace=True)
    # N!
    for j, q in enumerate(r):
        qc.crz(N * np.pi / (2**j), flag, q)
    # !!(N)
    qc.compose(QFT(m, do_swaps=False).inverse(), qubits=r, inplace=True)
    qc.cx(r[-1], flag)
#Computational basis
    qc.compose(QFT(m, do_swaps=False), qubits=r, inplace=True)
    qc.compose(QFT(m, do_swaps=False).inverse(), qubits=r, inplace=True)

# Circuit_MAD_gate
def _mad_gate(n_a: int, n_b: int, k: int, N: int):
    sub = QuantumCircuit(n_a + n_b + 2, name=f"MAD({k%N})")   # [a | b | extra | flag]
    a_sub   = [sub.qubits[i] for i in range(n_a)]
    b_sub   = [sub.qubits[n_a + j] for j in range(n_b)]
    extra   =  sub.qubits[n_a + n_b + 0]
    flag    =  sub.qubits[n_a + n_b + 1]
    mad_out_k_N(sub, a_sub, b_sub, [extra, flag], k % N, N)
    return sub.to_gate(label=f"MAD({k%N})")

def _mad_ctrl(qc: QuantumCircuit, a, b, aux, k: int, N: int, ctrl):
    gate = _mad_gate(len(a), len(b), k % N, N).control(1)
    qc.append(gate, [ctrl] + a + b + aux[:2])

def _mad_ctrl_inverse_of_invk(qc: QuantumCircuit, a, b, aux, inv_k: int, N: int, ctrl):
    gate_inv = _mad_gate(len(a), len(b), inv_k % N, N).inverse().control(1)
    qc.append(gate_inv, [ctrl] + a + b + aux[:2])

### res <- a^x mod N (MSB-1st)
def exp_a_mod_N_outplace(qc: QuantumCircuit, a_const: int, N: int, wires_x, wires_res, wires_support, wires_aux):
    qc.x(wires_res[0])                                    
    a_mod = a_const % N
    n = len(wires_x)
    for step in range(n):                             ## MSB -> LSB
        idx  = n - 1 - step
        ctrl = wires_x[idx]
        k_i  = pow(a_mod, 1 << idx, N)
        if k_i == 1:
            continue
        if np.gcd(k_i, N) != 1:
# Cleanup needs inv_k; as Pennylane
            continue
        inv_k = pow(k_i, -1, N)

        _mad_ctrl(qc, wires_res, wires_support, wires_aux, k_i, N, ctrl)
        for r, s in zip(wires_res, wires_support):
            qc.cswap(ctrl, r, s)
        _mad_ctrl_inverse_of_invk(qc, wires_res, wires_support, wires_aux, inv_k, N, ctrl)

# Circuit - measure res
def circuit_exp(a_const, x_val, N, wires_x, wires_res, wires_support, wires_aux, n_qubits=11):
    qc = QuantumCircuit(n_qubits, len(wires_res))
    q  = qc.qubits
    x   = [q[i] for i in wires_x]
    res = [q[i] for i in wires_res]
    sup = [q[i] for i in wires_support]
    aux = [q[i] for i in wires_aux]               

    for bit_pos, qb in enumerate(x):                      
        if (x_val >> bit_pos) & 1: 
            qc.x(qb)

    exp_a_mod_N_outplace(qc, a_const, N, x, res, sup, aux)

    for i, qb in enumerate(res):                          # measure res(LSB..MSB)
        qc.measure(qb, i)
    return qc

# Example
N = 7
a_const = 4
x_val   = 2
wires_x       = [0,1,2]
wires_res     = [3,4,5]
wires_support = [6,7,8]
wires_aux     = [9,10] 

qc = circuit_exp(a_const, x_val, N, wires_x, wires_res, wires_support, wires_aux, n_qubits=11)

backend = AerSimulator()
res = backend.run(transpile(qc, backend), shots=1, memory=True).result()
print(f"{a_const}^{x_val} mod {N} = {int(res.get_memory()[0], 2)}")
print(qc.draw('text'))

ops = qc.count_ops()
print(f"Width: {qc.num_qubits}")
print(f"Depth: {qc.depth()}")
print(f"Gate count (total): {sum(ops.values())}")

PART 2

Modular Exponentiation - Qiskit circuit library

In [117]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

#Qiskit ripple-carry adder
try:
    from qiskit.circuit.library import CDKMRippleCarryAdder as RippleCarryAdder
except ImportError:
    from qiskit.circuit.library import RippleCarryAdder

def _prepare_const_bits(qc: QuantumCircuit, regs, value: int):
    for i, q in enumerate(regs):
        if (value >> i) & 1:
            qc.x(q)

def _adder(n: int):
    return RippleCarryAdder(num_state_qubits=n, kind="fixed").to_gate()

#b <- (b + k) mod N
def mod_add_const(qc: QuantumCircuit, b, k: int, N: int, carry_add, carry_tmp, carry_ctrl, scratch_a, scratch_Nbar):
    n = len(b)
    k %= N
    if k == 0:  # no-op
        return

    add = _adder(n)
    add_c = add.control(1)

    # 1) b += k
    _prepare_const_bits(qc, scratch_a, k)
    qc.append(add, scratch_a + b + [carry_add])
    _prepare_const_bits(qc, scratch_a, k)

    # 2) carry_tmp := [b >= N], carry-out of b + (2^n - N)
    Nbar = (1 << n) - N
    _prepare_const_bits(qc, scratch_Nbar, Nbar)
    qc.append(add, scratch_Nbar + b + [carry_tmp])           
    # 3) if carry_tmp: b += (2^n - N)
    qc.append(add_c, [carry_tmp] + scratch_Nbar + b + [carry_ctrl])
    # 4) restores b and reset carry_tmp to |0>)
    qc.append(add.inverse(), scratch_Nbar + b + [carry_tmp])
    _prepare_const_bits(qc, scratch_Nbar, Nbar)

# sup + k*res (mod N)
def mult_const_outplace(qc: QuantumCircuit, res, sup, k: int, N: int, carries, scratch_a, scratch_Nbar):
    n = len(res)
    for i, ri in enumerate(res):                      # sup += sum_i (k * 2^i [ri]) mod N
        add_val = (k * (1 << i)) % N
        if add_val == 0:
            continue
        sub = QuantumCircuit(       # add by constant
            n + 3 + n + n, name=f"c+={add_val}"
        )
        # layout:
        b_sub = list(range(0, n))
        ca, ct, cc = n, n+1, n+2
        sa = list(range(n+3, n+3+n))
        sn = list(range(n+3+n, n+3+n+n))
        mod_add_const(sub, b_sub, add_val, N, ca, ct, cc, sa, sn)
        gate = sub.to_gate().control(1)
        qc.append(gate, [ri] + sup + carries + scratch_a + scratch_Nbar)

#cleanup
def cmult_const_outplace(qc: QuantumCircuit, ctrl, res, sup, k: int, N: int, carries, scratch_a, scratch_Nbar):
    n = len(res)
    k_mod = k % N
    if k_mod == 0:
        return
    inv_k = pow(k_mod, -1, N) if np.gcd(k_mod, N) == 1 else None

    # add one control
    sub = QuantumCircuit(2*n + 3 + n + n, name=f"mult({k_mod})")
    r = list(range(0, n))
    s = list(range(n, 2*n))
    ca, ct, cc = 2*n, 2*n+1, 2*n+2
    sa = list(range(2*n+3, 2*n+3+n))
    sn = list(range(2*n+3+n, 2*n+3+n+n))

    # s += k * r
    mult_const_outplace(sub, r, s, k_mod, N, [ca, ct, cc], sa, sn)
    # swap(r, s)
    for ri, si in zip(r, s):
        sub.swap(ri, si)
    # cleanup: s += (-inv_k) * r
    if inv_k is not None:
        mult_const_outplace(sub, r, s, (-inv_k) % N, N, [ca, ct, cc], sa, sn)

    gate = sub.to_gate().control(1)
    qc.append(gate, [ctrl] + res + sup + [*carries, *scratch_a, *scratch_Nbar])

# modular exponentiation
def modexp_library(qc: QuantumCircuit, a_const: int, N: int, x, res, sup, carries, scratch_a, scratch_Nbar):
    qc.x(res[0])                       #res = 1
    a_mod = a_const % N
    n = len(x)
    for step in range(n):              #MSB -> LSB
        idx = n - 1 - step
        k_i = pow(a_mod, 1 << idx, N)
        if k_i != 1:
            cmult_const_outplace(qc, x[idx], res, sup, k_i, N, carries, scratch_a, scratch_Nbar)

# Circuit
def circuit_modexp_library(a_const, x_val, N, wires_x, wires_res, wires_sup, wires_carries, wires_scratch_a, wires_scratch_Nbar, n_qubits):
    qc = QuantumCircuit(n_qubits, len(wires_res))
    q  = qc.qubits
    x   = [q[i] for i in wires_x]
    res = [q[i] for i in wires_res]
    sup = [q[i] for i in wires_sup]
    ca, ct, cc = [q[i] for i in wires_carries]
    sa  = [q[i] for i in wires_scratch_a]
    sn  = [q[i] for i in wires_scratch_Nbar]

    for i, qb in enumerate(x):         #|x>
        if (x_val >> i) & 1:
            qc.x(qb)

    modexp_library(qc, a_const, N, x, res, sup, [ca, ct, cc], sa, sn)

    for i, qb in enumerate(res):
        qc.measure(qb, i)
    return qc

# Examples
if __name__ == "__main__":
    backend = AerSimulator()

    #layout
    n = 3
    wires_x       = [0,1,2]
    wires_res     = [3,4,5]
    wires_sup     = [6,7,8]
    wires_carries = [9,10,11]       # add, tmp, ctrl
    wires_sa      = [12,13,14]      # scratch for constants
    wires_sn      = [15,16,17]      # scratch for (2^n - N)
    n_qubits      = 18
    

    # Example 1
    N = 7
    a_const = 4 
    x_val = 2
    qc = circuit_modexp_library(a_const, x_val, N, wires_x, wires_res, wires_sup, wires_carries, wires_sa, wires_sn, n_qubits)
    r = backend.run(transpile(qc, backend), shots=1, memory=True).result()
    got = int(r.get_memory()[0], 2); exp = pow(a_const % N, x_val, N)
    print(f"{a_const}^{x_val} mod {N} = {got} (expected {exp})")

    # Example 2
    N = 7
    a_const = 3 
    x_val = 2
    qc = circuit_modexp_library(a_const, x_val, N, wires_x, wires_res, wires_sup, wires_carries, wires_sa, wires_sn, n_qubits)
    r = backend.run(transpile(qc, backend), shots=1, memory=True).result()
    got = int(r.get_memory()[0], 2); exp = pow(a_const % N, x_val, N)
    print(f"{a_const}^{x_val} mod {N} = {got} (expected {exp})")
    
    # --- quick circuit stats ---
ops = qc.count_ops()
print(f"Width: {qc.num_qubits}")
print(f"Depth: {qc.depth()}")
print(f"Gate count (total): {sum(ops.values())}")



4^2 mod 7 = 2 (expected 2)
3^2 mod 7 = 2 (expected 2)
Width: 18
Depth: 5
Gate count (total): 8


Compare 

The gate count, and depth is lower on the Qiskit circuit library based modular exponentiation, while the width is lower on the QFT based modular exponentiation. Thus the QFT based, is much heavier and consists of more gates and depth, while the QISKIT circuit based has a lower gate count and and depth, as it is much simpler, but has higher number of qubits and width (as displayed as output).